In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l1.6 Optimization loop

> 🎯 **Goal:** Combine everything from this unit into a real training loop and train your first (tiny) language model end to end.

**Why this matters.** This is the payoff of the whole unit. Tensors, matmul, softmax, cross-entropy, and gradients were the parts; now we assemble them into the loop that actually learns. The model we train, a **bigram**, is the simplest possible language model, but the training loop you write here is the very same loop (forward, loss, backward, step) that trains models with billions of parameters. Learn it once at tiny scale and you understand it at every scale.

**The intuition.** A bigram model predicts the next character using only the current character, like guessing the next letter purely from the one you just saw. After the letter "q" it learns that "u" is overwhelmingly likely; after a space it learns common starting letters. It has no memory beyond one character, so it can never form real words, but it is enough to watch genuine learning happen. The training loop is a feedback machine: show the model some text, measure how surprised it was (loss), compute which way to nudge it (gradients), take a small step in that direction (optimizer), and repeat hundreds of times.

**The mechanics.** Walking the loop slowly, term by term:

- **Tokenizer:** `CharTokenizer` maps each character to an integer id and back, because models work on numbers, not letters.
- **Batch:** each step grabs a handful of random chunks of text at once (`batch` of them, each `block` characters long). Training on many examples per step is faster and steadier than one at a time. `xb` is the input characters; `yb` is the same text shifted by one, the "correct next character" answers.
- **Forward:** `model(xb, yb)` returns predictions and the cross-entropy `loss`.
- **`opt.zero_grad(set_to_none=True)`:** clears last step's gradients so they do not accumulate (the pitfall from the previous lesson).
- **`loss.backward()`:** autograd fills in every parameter's gradient.
- **`opt.step()`:** the optimizer (here **AdamW**, a smart, popular variant of gradient descent) nudges every parameter opposite its gradient, scaled by the learning rate `lr`.

We record each step's loss and plot it. The `steps` line just runs fewer iterations under continuous integration so automated tests stay fast; locally it runs the full 300. Watch the curve fall as the optimizer does its job.

In [ ]:
import matplotlib
matplotlib.use("Agg")  # headless backend (CI has no display)
import matplotlib.pyplot as plt
from torch.nn import functional as F
from pocketlm import CharTokenizer, BigramLM

corpus = open(CORPUS_PATH, encoding="utf-8").read()
tok = CharTokenizer.from_text(corpus)
data = torch.tensor(tok.encode(corpus), dtype=torch.long)

torch.manual_seed(0)
model = BigramLM(tok.vocab_size)
opt = torch.optim.AdamW(model.parameters(), lr=0.1)
block, batch = 32, 32
steps = 50 if os.environ.get("POCKETLM_CI") else 300
losses = []
for _ in range(steps):
    ix = torch.randint(len(data) - block - 1, (batch,))
    xb = torch.stack([data[i:i + block] for i in ix])
    yb = torch.stack([data[i + 1:i + 1 + block] for i in ix])
    _, loss = model(xb, yb)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
    losses.append(loss.item())
plt.plot(losses)
plt.xlabel("step")
plt.ylabel("loss")
plt.title("Bigram training loss")
plt.show()
print("first loss:", round(losses[0], 3), "last loss:", round(losses[-1], 3))

**What just happened.** The printed loss dropped from its first value to a noticeably lower last value, and the plotted curve slopes downward. That descending line is learning made visible: with each step the optimizer made the model a little less surprised by real text. The model is figuring out which character tends to follow which.

Next we **sample** from it: start with a seed token, ask the model for next-character probabilities, draw one with `torch.multinomial` (which picks a token at random weighted by its probability), append it, and repeat. The result is character soup with roughly the right letter frequencies and believable letter pairs, but no real words. That is the hard ceiling of a bigram, since it only ever looks back one character. The entire rest of the course is about lifting that ceiling by giving the model more context and more capacity.

In [ ]:
idx = torch.zeros((1, 1), dtype=torch.long)
for _ in range(120):
    logits, _ = model(idx[:, -1:])
    probs = F.softmax(logits[:, -1, :], dim=-1)
    idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
print(tok.decode(idx[0].tolist()))

In [ ]:
# The optimizer actually reduced the loss.
assert sum(losses[-5:]) / 5 < losses[0]

**What just happened.** The final assert confirms the average of the last five losses is below the very first loss, mechanical proof that the optimizer truly reduced the loss rather than just bouncing around. You have now trained a language model from scratch: tokenized text, batched it, run forward and backward passes, stepped an optimizer, and sampled new text from the trained weights. Every later unit reuses this exact skeleton.

⚠️ **Common pitfalls**
- Forgetting `opt.zero_grad()` inside the loop. Gradients would accumulate across steps and training would diverge instead of converge. It is here on purpose; never delete it.
- Misreading bigram output. Expecting real words from a one-character-of-context model is a setup for disappointment. Character soup with sensible letter frequencies is success at this stage, not failure.
- Learning rate too high or too low. A huge `lr` makes the loss explode or oscillate wildly; a tiny one makes it crawl. The smooth downward curve you saw means `lr=0.1` is in the sweet spot for this tiny model.

🏋️ **Try it yourself.** Change `lr=0.1` to `lr=1.0` and rerun, then try `lr=0.001`. With the large rate the loss curve gets jagged or worse; with the tiny rate it barely moves over 300 steps. Seeing both failure modes builds the intuition for why picking a learning rate is one of the most important decisions in training, a theme that returns throughout the course.